In [ ]:
# ====================================
# Import libraries
# ====================================
import torch
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import random_split, DataLoader
import torchvision.transforms.v2 as T
from torch.utils.data import TensorDataset
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random

#Nb. I have tried to re-use the code for the classification task as much
#as I could for consistency. I have flagged the spots where something changes with "NEW".

In [11]:
# ====================================
# Reproducibility and device
# ====================================
r_seed = 99
generator = torch.Generator()

def reset_seed(seed=r_seed):
    """Reset all seeds at once."""
    random.seed(seed) #reset python random seed
    np.random.seed(seed) #reset numpy random seed
    torch.manual_seed(seed) #reset pytorch random seed
    generator.manual_seed(seed) #reset the dataloader random seed
    if torch.cuda.is_available(): #if we use the gpu
        torch.cuda.manual_seed_all(seed)

reset_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


In [12]:
# ====================================
# Load the Clock dataset (75x75)
# ====================================
img_size = 75 #So that we can change it to 150x150 if needed
images_path = f"data/{img_size:03d}_images.npy"
labels_path = f"data/{img_size:03d}_labels.npy"

images_np = np.load(images_path)
labels_np = np.load(labels_path)

#Check out our images
print("images:", images_np.shape, images_np.dtype, images_np.min(), images_np.max()) #18'000 images, 75x75
print("labels:", labels_np.shape, labels_np.dtype) #18'000 labels, 2 dimensions (hour, min)
print("hours:", np.unique(labels_np[:, 0])) #they go from 0 to 11
print("minutes:", labels_np[:, 1].min(), labels_np[:, 1].max()) #they go from 0 to 59

#Convert into pytorch tensors
images = torch.from_numpy(images_np).float() #turn numpy array into pytorch tensor, float
images = images.unsqueeze(1) #add channel dimension, we need format (batch_size, channels, height, width) (greyscale = 1 channel)
mean = images.mean()
std = images.std()
images = (images - mean) / std #standardize the pixels (easier for training)

labels_hours = torch.from_numpy(labels_np[:, 0] + labels_np[:, 1] / 60.0).float() #NEW: mapping "03:00" → 3.0 and "05:30" → 5.5).
print(labels_hours.shape, labels_hours.min(), labels_hours.max()) #18'000 labels, 1 dimension (min). Minutes go from 0 (00:00) to 719 (11:59). 

#What's new:
#label_minutes is now a float number (e.g. 3.5 hours)

images: (18000, 75, 75) uint8 16 255
labels: (18000, 2) int64
hours: [ 0  1  2  3  4  5  6  7  8  9 10 11]
minutes: 0 59
torch.Size([18000]) tensor(0.) tensor(11.9833)


In [ ]:
# =====================================
# Data Agumentation
# =====================================

#The images above appear:
# - Rotated in different orientations
# - There are bright spots
# - They are slightly tilted
#We need to try to account for these differences

#train_transform = T.Compose([
#    T.RandomRotation(degrees=180),                          #rotation
#    T.RandomPerspective(distortion_scale=0.15, p=0.4),       #tilt
#    T.ColorJitter(brightness=0.3, contrast=0.3),             #soften the bright spots
#])

In [13]:
#=====================================
# Train-Validation-Test split
# ====================================
reset_seed()

dataset = TensorDataset(images, labels_hours)
n = len(dataset)

train_size = int(0.8 * n)
valid_size = int(0.1 * n)
test_size = n - train_size - valid_size

train_data, valid_data, test_data = random_split(
    dataset,
    [train_size, valid_size, test_size],
    generator=generator
)

batch_size = 64

train_dataloader = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True,
    generator=generator
)

valid_dataloader = DataLoader(
    valid_data,
    batch_size=batch_size,
    shuffle=True 
)

test_dataloader = DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=True
)

In [ ]:
# =====================================
# Setting up the CNN for regression
# =====================================

class Net_4conv(nn.Module):   #Final CNN
    def __init__(self):  #allow to provide a desired number of classes
        super().__init__()
        #Input image (1, 75, 75) (greyscale, 75x75)
        self.conv1 = nn.Conv2d(1, 6, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 3, padding=1)
        self.conv3 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv4 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc1 = nn.Linear(64 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 1) #NEW: now we want only 1 output value

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.pool(F.relu(self.conv4(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
#What's new?
#Now our NN needs to output one number, not n (num_classes) classes.

In [ ]:
# ======================================================
# Defining the functions for training and evaluation
# ======================================================


#Train the model

def train_model(model, train_dataloader, optimizer, criterion, epochs): #NEW: no more bin_size
    
    """Train a model for a given number of epochs."""
    
    for epoch in range(epochs):
        
        model.train() 
        running_loss = 0.0
        
        for inputs, hours in train_dataloader: 
            
            #data agumentation
            #inputs = train_transform(inputs)
            
            #To use external GPU
            inputs = inputs.to(device)                    
            hours = hours.to(device)
            
            #zero the parameter gradients
            optimizer.zero_grad()

            #forward + backward + optimize
            outputs = model(inputs).squeeze(1) #NEW: these are the new predictions
            loss = criterion(outputs, hours) #NEW: regression loss  
            loss.backward() #compute gradients
            optimizer.step() #optimize weights

            #compute running loss
            running_loss += loss.item()
        print(
            f"Epoch {epoch + 1}/{epochs}: "
            f"- Loss = {running_loss/len(train_dataloader):.4f}"
        )

#Validate the model

def evaluate_model(model, dataloader): #NEW: no more bin_size

    """Evaluate a model using the average circular time error."""

    model.eval()

    total_error = 0
    total = 0

    with torch.no_grad():
        for inputs, hours in dataloader:
            
            #To use external GPU
            inputs = inputs.to(device)                    
            hours = hours.to(device)            
            
            #Get predictions (in total number of minutes)
            outputs = model(inputs).squeeze(1) #NEW: predictions (in hours, 3.5!!!)
            predicted_hours = torch.remainder(outputs, 12.0) #NEW: restrict prediction in (0, 720) minutes range

            #Calculate absolute difference and the circular difference (in minutes!!)
            predicted_minutes = predicted_hours * 60.0 #We re-convert them in minutes to make it comparable with the other models
            true_minutes = hours * 60.0 #It doesn't affect the learning process
            difference = torch.abs(predicted_minutes - true_minutes) 
            circular_difference = torch.minimum(difference, 720 - difference)

            #Compute the errors
            total_error += circular_difference.sum().item()
            total += hours.size(0)

    common_sense_error = total_error / total

    return common_sense_error

In [ ]:
# =====================================
# Regression experiments - to choose lr and epochs
# =====================================

experiments = [
    {"learning_rate": 0.001, "epochs": 10},
    {"learning_rate": 0.001, "epochs": 20},
    {"learning_rate": 0.0005, "epochs": 10},
    {"learning_rate": 0.0005, "epochs": 20}
]

results = []

for experiment in experiments:

    lr = experiment["learning_rate"]
    epochs = experiment["epochs"]

    print(f"\nExperiment: learning rate = {lr}, epochs = {epochs}")

    #New seed for every experiment
    reset_seed()

    #New model for every experiment
    model = Net_4conv().to(device)

    #Loss function
    criterion = nn.MSELoss()

    #Optimizer
    optimizer = optim.Adam(model.parameters(), lr=lr)

    #Train the model
    train_model(
        model,
        train_dataloader,
        optimizer,
        criterion,
        epochs
    )

    #Validate the model
    valid_error = evaluate_model(model, valid_dataloader)

    #Store results
    results.append({
        "Learning rate": lr,
        "Epochs": epochs,
        "Validation error (min)": valid_error
    })

    print(f"Validation error: {valid_error:.2f} minutes")

results_df = pd.DataFrame(results)
print(results_df)



Experiment: learning rate = 0.001, epochs = 10
Epoch 1/10: - Loss = 12.8997
Epoch 2/10: - Loss = 12.1090
Epoch 3/10: - Loss = 12.0460
Epoch 4/10: - Loss = 11.7233
Epoch 5/10: - Loss = 10.9843
Epoch 6/10: - Loss = 9.9978
Epoch 7/10: - Loss = 9.5090
Epoch 8/10: - Loss = 9.2896
Epoch 9/10: - Loss = 8.8443
Epoch 10/10: - Loss = 8.5637
Validation error: 133.39 minutes

Experiment: learning rate = 0.001, epochs = 20
Epoch 1/20: - Loss = 12.8997
Epoch 2/20: - Loss = 12.1090
Epoch 3/20: - Loss = 12.0460
Epoch 4/20: - Loss = 11.7233
Epoch 5/20: - Loss = 10.9843
Epoch 6/20: - Loss = 9.9978
Epoch 7/20: - Loss = 9.5090
Epoch 8/20: - Loss = 9.2896
Epoch 9/20: - Loss = 8.8443
Epoch 10/20: - Loss = 8.5637
Epoch 11/20: - Loss = 8.2549
Epoch 12/20: - Loss = 7.9102
Epoch 13/20: - Loss = 7.5915
Epoch 14/20: - Loss = 7.1775
Epoch 15/20: - Loss = 6.8369
Epoch 16/20: - Loss = 6.4099
Epoch 17/20: - Loss = 5.9658
Epoch 18/20: - Loss = 5.4962
Epoch 19/20: - Loss = 5.1959
Epoch 20/20: - Loss = 4.8821
Validatio

In [ ]:
# =====================================
# Regression experiments - to choose Loss Function
# =====================================

#Loss function
loss_functions = {
    "MSELoss": nn.MSELoss(),
    "L1Loss": nn.L1Loss(),
    "SmoothL1Loss": nn.SmoothL1Loss(),     
}
best_epochs, best_lr = 20, 0.001
results, models = [], {}

for loss_name, criterion in loss_functions.items():

    print(f"\nTraining with {loss_name}")

    #New seed for every experiment
    reset_seed()

    #New model for each experiment
    model = Net_4conv().to(device)

    #Optimizer
    optimizer = optim.Adam(model.parameters(), lr=best_lr)

    #Train the model
    train_model(
        model,
        train_dataloader,
        optimizer,
        criterion,
        best_epochs
    )

    #Validate the model
    validation_error = evaluate_model(
        model, 
        valid_dataloader
    )

    models[loss_name] = model

    #Store results
    results.append({
        "Loss function": loss_name,
        "Validation error (minutes)": validation_error
    })
    pd.DataFrame(results).to_csv("results_regress.csv", index=False)
    print(f"{loss_name} validation error: {validation_error:.2f} minutes")

loss_df = pd.DataFrame(results)


Best: learning rate = 0.001, epochs = 10

Training with MSELoss
Epoch 1/20: - Loss = 12.8997
Epoch 2/20: - Loss = 12.1090
Epoch 3/20: - Loss = 12.0460
Epoch 4/20: - Loss = 11.7233
Epoch 5/20: - Loss = 10.9843
Epoch 6/20: - Loss = 9.9978
Epoch 7/20: - Loss = 9.5090
Epoch 8/20: - Loss = 9.2896
Epoch 9/20: - Loss = 8.8443
Epoch 10/20: - Loss = 8.5637
Epoch 11/20: - Loss = 8.2549
Epoch 12/20: - Loss = 7.9102
Epoch 13/20: - Loss = 7.5915
Epoch 14/20: - Loss = 7.1775
Epoch 15/20: - Loss = 6.8369
Epoch 16/20: - Loss = 6.4099
Epoch 17/20: - Loss = 5.9658
Epoch 18/20: - Loss = 5.4962
Epoch 19/20: - Loss = 5.1959
Epoch 20/20: - Loss = 4.8821
MSELoss validation error: 103.82 minutes

Training with L1Loss
Epoch 1/20: - Loss = 3.0984
Epoch 2/20: - Loss = 3.0185
Epoch 3/20: - Loss = 2.9937
Epoch 4/20: - Loss = 2.8496
Epoch 5/20: - Loss = 2.6460
Epoch 6/20: - Loss = 2.4933
Epoch 7/20: - Loss = 2.3830
Epoch 8/20: - Loss = 2.2648
Epoch 9/20: - Loss = 2.1475
Epoch 10/20: - Loss = 2.0643
Epoch 11/20: - 

In [ ]:
# =====================================
# Test set - With Best Parameters
# =====================================

#Best parameters (according to validation results)
best_lr = 0.001
best_epochs = 20

best_loss = results_df.loc[results_df["Validation error (min)"].idxmin(), "Loss function"]

#Evaluate the final model on test set
test_error = evaluate_model(
    models[best_loss], 
    test_dataloader
)

print(f"Best loss: {best_loss}, test error: {test_error:.2f} minutes")

In [ ]:
# =====================================
# Questions
# =====================================

"""
What loss functions (nn.MSELoss, nn.L1Loss, nn.SmoothL1Loss) are appropriate?

- nn.MSELoss: computes the mean squared error between predictions and target y.

- nn.L1Loss (MAELoss): computes the mean absolute error between predictions and target y.

- nn.SmoothL1Loss: combines MSELoss and L1Loss by using a transition threshold (beta). For errors smaller 
then delta we use the MSELoss, for errors larger than beta we use the MAELoss.

The MSE loss penalises strongly the magnitude of the errors by taking their squared value, therefore large errors 
get a very high penalty. This also mean that if there is a very bad picture, that gives a huge error it will severely affect
the training process. MAE, on the other hand, penalises every minute of error equally, and is therefore less sensitive to outliers.
The Smooth Loss is an option in the middle: it is robust to big errors like MSE, but it's less aggressive. 

What boundary discontinuity issues arise around 12:00 / 0:00?

The problem that arises at the boundary 12:00 to 00:00 is the fact that two predictions such as 11:59 
and 00:02 are only 3 minutes apart, but when we convert them into total minutes 11:59 = 11*60 + 59 = 719 minutes
and 00:02 = 0*60 + 2 = 2 minutes they seem to be 719 - 2 = 717 minutes apart. 

This is due to the circular nature of time. We handled this problem in the validation function by taking
circular_difference = torch.minimum(difference, 720 - difference) before we compute the "common sense error".
However, the problem is that the loss function does not know this and penalises this misclassification as 717 minutes off.

"""